In [1]:
import h5torch
import os
cell_line_selection = ['GM12878', 'K562', 'HepG2', 'A549']
# Get all h5t files from the specified folder
h5t_files = [os.path.join('/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr', f) 
             for f in os.listdir('/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr') 
             if f.endswith('.h5t') and f.split('.')[0] in cell_line_selection]

In [2]:
h5t_files

['/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/GM12878.h5t',
 '/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/HepG2.h5t',
 '/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/K562.h5t',
 '/data/home/natant/Negatives/Data/Encode690/ENCODE_hg38_subset_101bp_celltypes_ATAC_H5_all_chr/A549.h5t']

In [3]:
TFs_per_cell_line = {}
for h5t_file in h5t_files:
    with h5torch.File(h5t_file, 'r') as h5f:
        #print(h5f["0/prot_names"][:].astype(str))
        cell_line = os.path.basename(h5t_file).split('.')[0]
        TFs_per_cell_line[cell_line] = [tf for tf in h5f["0/prot_names"][:].astype(str).tolist() if tf != 'ATAC_peak']

In [4]:
# Convert each TF list to a set for easier comparison
tf_sets = {cell_line: set(tfs) for cell_line, tfs in TFs_per_cell_line.items()}

# Find TFs present in all cell lines (complete overlap)
common_tfs_all = set.intersection(*tf_sets.values())
print(f"TFs common to all cell lines ({len(common_tfs_all)}): {sorted(common_tfs_all)}")

# Find pairwise overlaps
print("\nPairwise overlaps:")
cell_lines = list(TFs_per_cell_line.keys())
for i in range(len(cell_lines)):
    for j in range(i+1, len(cell_lines)):
        overlap = tf_sets[cell_lines[i]] & tf_sets[cell_lines[j]]
        print(f"{cell_lines[i]} ∩ {cell_lines[j]}: {len(overlap)} TFs")

# Find TFs unique to each cell line
print("\nUnique TFs per cell line:")
for cell_line in cell_lines:
    unique_tfs = tf_sets[cell_line] - set.union(*[tf_sets[cl] for cl in cell_lines if cl != cell_line])
    print(f"{cell_line}: {len(unique_tfs)} unique TFs - {sorted(unique_tfs) if unique_tfs else 'None'}")

TFs common to all cell lines (7): ['ATF3', 'CTCF', 'ELF1_(SC-631)', 'Max', 'USF-1', 'YY1_(SC-281)', 'ZBTB33']

Pairwise overlaps:
GM12878 ∩ HepG2: 21 TFs
GM12878 ∩ K562: 27 TFs
GM12878 ∩ A549: 10 TFs
HepG2 ∩ K562: 22 TFs
HepG2 ∩ A549: 11 TFs
K562 ∩ A549: 9 TFs

Unique TFs per cell line:
GM12878: 6 unique TFs - ['ATF2_(SC-81188)', 'FOXM1_(SC-502)', 'IKZF1_(IkN)_(UCLA)', 'Pbx3', 'ZEB1_(SC-25388)', 'ZZZ3']
HepG2: 6 unique TFs - ['ARID3A_(NB100-279)', 'CEBPD_(SC-636)', 'HSF1', 'IRF3', 'MYBL2_(SC-81192)', 'TCF7L2']
K562: 5 unique TFs - ['ATF1_(06-325)', 'Bach1_(sc-14700)', 'FOSL1_(SC-183)', 'NR2F2_(SC-271940)', 'SETDB1']
A549: 1 unique TFs - ['CREB1_(SC-240)']


In [5]:
GM12878_overlaps = {}
# Find pairwise overlaps
print("\nPairwise overlaps:")
cell_lines = list(TFs_per_cell_line.keys())
for cell_line in ['K562', 'HepG2', 'A549']:
    overlap = tf_sets['GM12878'] & tf_sets[cell_line]
    GM12878_overlaps[cell_line] = overlap
    print(f"GM12878 ∩ {cell_line}: {len(overlap)} TFs")




Pairwise overlaps:
GM12878 ∩ K562: 27 TFs
GM12878 ∩ HepG2: 21 TFs
GM12878 ∩ A549: 10 TFs


In [6]:
import os
import pandas as pd
import re
convert_dict = {
    "dinucl_sampled": "dinucl-sampled",
    "dinucl_shuffled": "dinucl-shuffled",
}
# Define the folder path
folder_path = "/data/home/natant/Negatives/Runs/Review_sane_model/other_cell_lines"

# Get all files ending in .ckpt
ckpt_files = [f for f in os.listdir(folder_path) if f.endswith('.ckpt')]

# Parse each filename to extract information
data = []
negative_types = ["dinucl-sampled", "celltype", "shuffled", "dinucl-shuffled", "neighbors"]
for filename in ckpt_files:   
    parts = filename.split('_')

    # Extract fields from the new naming scheme:
    # CT-GM12878, TF-ELK1$(1277-1), NEG-dinucl$shuffled, CV-5, ...
    cellline = None
    tf = None
    neg_type = None
    cv_split = None

    for p in parts:
        if p.startswith('CT-'):
            cellline = p[len('CT-'):]
        elif p.startswith('TF-'):
            # restore original underscores inside TF name
            tf = p[len('TF-'):].replace('$', '_')
        elif p.startswith('NEG-'):
            # keep '-' for later convert_dict mapping, but restore '_' inside mode
            neg_type = p[len('NEG-'):].replace('$', '_')
        elif p.startswith('CV-'):
            cv_split = p[len('CV-'):]



    
    
    data.append({
        'filename': filename,
        'file_path': os.path.join(folder_path, filename),
        'cellline': cellline,
        'TF': tf,
        'negative_type': neg_type,
        'cv_split': cv_split
    })

# Create DataFrame
df_ckpt = pd.DataFrame(data)
df_ckpt

,filename,file_path,cellline,TF,negative_type,cv_split
0,CT-GM12878_TF-NF-YA_NEG-celltype_CV-0_LR-0.000...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,NF-YA,celltype,0
1,CT-GM12878_TF-Pbx3_NEG-neighbors_CV-5_LR-0.000...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,Pbx3,neighbors,5
2,CT-GM12878_TF-YY1$(SC-281)_NEG-HQ_CV-5_LR-0.00...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,YY1_(SC-281),HQ,5
3,CT-K562_TF-JunD_NEG-shuffled_CV-5_LR-0.0002_Da...,/data/home/natant/Negatives/Runs/Review_sane_m...,K562,JunD,shuffled,5
4,CT-HepG2_TF-Nrf1_NEG-neighbors_CV-0_LR-0.0002_...,/data/home/natant/Negatives/Runs/Review_sane_m...,HepG2,Nrf1,neighbors,0
...,...,...,...,...,...,...
4262,CT-K562_TF-FOSL1$(SC-183)_NEG-dinucl$sampled_C...,/data/home/natant/Negatives/Runs/Review_sane_m...,K562,FOSL1_(SC-183),dinucl_sampled,4
4263,CT-HepG2_TF-RXRA_NEG-neighbors_CV-3_LR-0.0002_...,/data/home/natant/Negatives/Runs/Review_sane_m...,HepG2,RXRA,neighbors,3
4264,CT-K562_TF-ATF1$(06-325)_NEG-dinucl$sampled_CV...,/data/home/natant/Negatives/Runs/Review_sane_m...,K562,ATF1_(06-325),dinucl_sampled,0
4265,CT-GM12878_TF-NF-YB_NEG-dinucl$sampled_CV-1_LR...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,NF-YB,dinucl_sampled,1


In [7]:
total_runs = 0
for cell_line in GM12878_overlaps:
    for tf in GM12878_overlaps[cell_line]:
        selected_runs = df_ckpt[(df_ckpt['cellline'] == 'GM12878') & (df_ckpt['TF'] == tf)]
        print(f"TF: {tf}, Runs found: {len(selected_runs)}")
        total_runs += len(selected_runs)
print(f"Total runs found: {total_runs}")

TF: SRF, Runs found: 36
TF: TBP, Runs found: 36
TF: NF-YB, Runs found: 36
TF: Max, Runs found: 36
TF: STAT5A_(SC-74442), Runs found: 36
TF: SIX5, Runs found: 36
TF: ZBTB33, Runs found: 36
TF: Znf143_(16618-1-AP), Runs found: 36
TF: ELK1_(1277-1), Runs found: 36
TF: RFX5_(200-401-194), Runs found: 36
TF: ATF3, Runs found: 36
TF: ZNF274, Runs found: 36
TF: JunD, Runs found: 36
TF: USF-1, Runs found: 36
TF: Nrf1, Runs found: 36
TF: MAZ_(ab85725), Runs found: 36
TF: CEBPB_(SC-150), Runs found: 36
TF: NF-YA, Runs found: 36
TF: CTCF, Runs found: 36
TF: USF2, Runs found: 36
TF: SP1, Runs found: 36
TF: Egr-1, Runs found: 36
TF: ELF1_(SC-631), Runs found: 36
TF: ETS1, Runs found: 36
TF: Mxi1_(AF4185), Runs found: 36
TF: MEF2A, Runs found: 36
TF: YY1_(SC-281), Runs found: 36
TF: SRF, Runs found: 36
TF: NFIC_(SC-81335), Runs found: 36
TF: TBP, Runs found: 36
TF: Max, Runs found: 36
TF: ZBTB33, Runs found: 36
TF: RXRA, Runs found: 36
TF: RFX5_(200-401-194), Runs found: 36
TF: ATF3, Runs found: 36


In [8]:
all_runs = []
for cell_line in GM12878_overlaps:
    for tf in GM12878_overlaps[cell_line]:
        selected_runs = df_ckpt[(df_ckpt['cellline'] == 'GM12878') & (df_ckpt['TF'] == tf) & (df_ckpt['negative_type'] != 'HQ')]
        selected_runs["cellline_test"] = cell_line
        all_runs.append(selected_runs)
all_runs_df = pd.concat(all_runs, ignore_index=True)
all_runs_df

/tmp/ipykernel_1011511/4105315406.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_runs["cellline_test"] = cell_line
/tmp/ipykernel_1011511/4105315406.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_runs["cellline_test"] = cell_line
/tmp/ipykernel_1011511/4105315406.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/p

,filename,file_path,cellline,TF,negative_type,cv_split,cellline_test
0,CT-GM12878_TF-SRF_NEG-neighbors_CV-2_LR-0.0001...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,SRF,neighbors,2,K562
1,CT-GM12878_TF-SRF_NEG-shuffled_CV-2_LR-0.0001_...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,SRF,shuffled,2,K562
2,CT-GM12878_TF-SRF_NEG-dinucl$sampled_CV-3_LR-0...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,SRF,dinucl_sampled,3,K562
3,CT-GM12878_TF-SRF_NEG-shuffled_CV-5_LR-0.0001_...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,SRF,shuffled,5,K562
4,CT-GM12878_TF-SRF_NEG-dinucl$shuffled_CV-4_LR-...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,SRF,dinucl_shuffled,4,K562
...,...,...,...,...,...,...,...
1735,CT-GM12878_TF-YY1$(SC-281)_NEG-shuffled_CV-2_L...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,YY1_(SC-281),shuffled,2,A549
1736,CT-GM12878_TF-YY1$(SC-281)_NEG-dinucl$shuffled...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,YY1_(SC-281),dinucl_shuffled,2,A549
1737,CT-GM12878_TF-YY1$(SC-281)_NEG-dinucl$shuffled...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,YY1_(SC-281),dinucl_shuffled,4,A549
1738,CT-GM12878_TF-YY1$(SC-281)_NEG-celltype_CV-5_L...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,YY1_(SC-281),celltype,5,A549


In [19]:
all_runs_df.to_pickle('/data/home/natant/Negatives/TFBS_negatives/testing_ground/REVIEW_cross_cell_perf/20251218_all_cross_cell_runs.pkl')

In [11]:
all_runs_HQ = []
for cell_line in GM12878_overlaps:
    for tf in GM12878_overlaps[cell_line]:
        selected_runs = df_ckpt[(df_ckpt['cellline'] == 'GM12878') & (df_ckpt['TF'] == tf) & (df_ckpt['negative_type'] == 'HQ')]
        selected_runs["cellline_test"] = cell_line
        all_runs_HQ.append(selected_runs)
all_runs_HQ_df = pd.concat(all_runs_HQ, ignore_index=True)
all_runs_HQ_df

/tmp/ipykernel_1011511/3213720869.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_runs["cellline_test"] = cell_line
/tmp/ipykernel_1011511/3213720869.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_runs["cellline_test"] = cell_line
/tmp/ipykernel_1011511/3213720869.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/p

,filename,file_path,cellline,TF,negative_type,cv_split,cellline_test
0,CT-GM12878_TF-SRF_NEG-HQ_CV-3_LR-0.0002_Date-2...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,SRF,HQ,3,K562
1,CT-GM12878_TF-SRF_NEG-HQ_CV-1_LR-0.0002_Date-2...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,SRF,HQ,1,K562
2,CT-GM12878_TF-SRF_NEG-HQ_CV-0_LR-0.0002_Date-2...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,SRF,HQ,0,K562
3,CT-GM12878_TF-SRF_NEG-HQ_CV-2_LR-0.0002_Date-2...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,SRF,HQ,2,K562
4,CT-GM12878_TF-SRF_NEG-HQ_CV-5_LR-0.0002_Date-2...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,SRF,HQ,5,K562
...,...,...,...,...,...,...,...
343,CT-GM12878_TF-YY1$(SC-281)_NEG-HQ_CV-2_LR-0.00...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,YY1_(SC-281),HQ,2,A549
344,CT-GM12878_TF-YY1$(SC-281)_NEG-HQ_CV-4_LR-0.00...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,YY1_(SC-281),HQ,4,A549
345,CT-GM12878_TF-YY1$(SC-281)_NEG-HQ_CV-1_LR-0.00...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,YY1_(SC-281),HQ,1,A549
346,CT-GM12878_TF-YY1$(SC-281)_NEG-HQ_CV-0_LR-0.00...,/data/home/natant/Negatives/Runs/Review_sane_m...,GM12878,YY1_(SC-281),HQ,0,A549


In [17]:
all_runs_HQ_df.to_pickle('/data/home/natant/Negatives/TFBS_negatives/testing_ground/REVIEW_cross_cell_perf/20251218_all_cross_cell_runs_HQ.pkl')

In [12]:
all_runs_df['file_path'].unique().shape[0]

900

In [13]:
all_runs_HQ_df['file_path'].unique().shape[0]

180

In [14]:
len(ckpt_files)

4267

In [15]:
# total checkpoint files for GM12878
print(df_ckpt["cellline"].value_counts())
# total unique checkpoints used for cross-cell evaluation
print(all_runs_df['file_path'].unique().shape[0] + all_runs_HQ_df['file_path'].unique().shape[0])

cellline
GM12878    1296
K562       1296
HepG2      1171
A549        504
Name: count, dtype: int64
1080


So these missing checkpoint just don't have any overlapping TFs with the other cell lines?

In [16]:
all_runs_df = pd.read_pickle('/data/home/natant/Negatives/TFBS_negatives/testing_ground/REVIEW_cross_cell_perf/20251218_all_cross_cell_runs.pkl')
all_runs_df

FileNotFoundError: [Errno 2] No such file or directory: '/data/home/natant/Negatives/TFBS_negatives/testing_ground/REVIEW_cross_cell_perf/20251218_all_cross_cell_runs.pkl'

In [28]:
run_tuples = [(row['file_path'], row['TF'], row['negative_type'], row['cv_split'], row['cellline_test']) 
              for _, row in all_runs_df.iterrows()]

In [ ]:
queue = Queue()

# Populate the queue with cell_tf_neg_combinations
for combination in run_tuples:
    queue.put(combination)

# Function to process combinations from the queue
def worker():
    while not queue.empty():
        file_path,tf, neg_mode, cv, cell_type = queue.get()
        command = [
            "python", 
            "/data/home/natant/Negatives/testing_ground/20251118_test_ckpt.py",
            "--ckpt_path", file_path,
            "--datafolder", datafolder,
            "--TF", tf, 
            "--celltype", cell_type, 
            "--neg_mode", neg_mode, 
            "--devices", "1",
            "--cross_val_set", str(cv),
            "--batch_size", "256",
            "--group_name", group_name
        ]
        subprocess.run(command)
        queue.task_done()

# Create and start threads
threads = []
for _ in range(max_concurrent_models):
    t = Thread(target=worker)
    t.start()
    threads.append(t)

# Wait for all threads to finish
for t in threads:
    t.join()